# Sprint 4: Prompt Augmentation and Response Generation

This notebook demonstrates the full RAG response flow:
1. Query retrieval
2. Prompt augmentation
3. LLM response generation
4. Dataset-based evaluation

In [7]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
import urllib.request
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return start


ROOT = find_repo_root(Path.cwd())
SRC_DIR = ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

PORT = 8017
BASE_URL = f"http://127.0.0.1:{PORT}"


def post_json(path: str, payload: dict, timeout: int = 120):
    req = urllib.request.Request(
        f"{BASE_URL}{path}",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.status, json.loads(resp.read().decode("utf-8"))


def get_json(path: str, timeout: int = 60):
    with urllib.request.urlopen(f"{BASE_URL}{path}", timeout=timeout) as resp:
        return resp.status, json.loads(resp.read().decode("utf-8"))


print("ROOT:", ROOT)
print("SRC_DIR on sys.path:", str(SRC_DIR) in sys.path)

ROOT: c:\Users\mlamar\Working Directory(Local)\Forge\m2\da-rag-project-2026_mlamar2019_2
SRC_DIR on sys.path: True


## Authenticate and Capture Token

In [8]:
from datetime import datetime

from ailab.utils.azure import authenticate_ailab_interactive

auth_result = authenticate_ailab_interactive()
os.environ["AILAB_BEARER_TOKEN"] = auth_result["token"]

expires_on = datetime.fromtimestamp(auth_result["expires_on"])
print("Authenticated via:", auth_result["auth_method"])
print("Token expires on:", expires_on)

Authenticated via: interactive_browser
Token expires on: 2026-03-24 16:05:09


## Start FastAPI Server

In [9]:
cmd = [sys.executable, "-m", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", str(PORT)]
server_env = dict(os.environ)
server_env["PYTHONPATH"] = str(SRC_DIR)

server_proc = subprocess.Popen(cmd, cwd=str(ROOT), env=server_env)

ready = False
for _ in range(40):
    try:
        status_code, _ = get_json("/health", timeout=2)
        if status_code == 200:
            ready = True
            break
    except Exception:
        time.sleep(0.5)

print("Server ready:", ready)

Server ready: True


## Ingest Dataset (Small Batch)

In [10]:
status_code, ingest_result = post_json(
    "/vectordb/ingest",
    {
        "source": "hf://datasets/rag-datasets/rag-mini-wikipedia/data/passages.parquet/part.0.parquet",
        "limit": 8,
    },
    timeout=300,
)
print("Status:", status_code)
print(json.dumps(ingest_result, indent=2)[:1000])


Status: 200
{
  "ingested_count": 8,
  "source": "hf://datasets/rag-datasets/rag-mini-wikipedia/data/passages.parquet/part.0.parquet",
  "status": {
    "initialized": true,
    "document_count": 8,
    "index_info": "LlamaIndex VectorStoreIndex"
  }
}


## Run End-to-End RAG Query

In [11]:
query_payload = {"query": "What is artificial intelligence?", "top_k": 3}
status_code, rag_result = post_json("/rag/query", query_payload, timeout=180)
print("Status:", status_code)
print("Answer:", rag_result.get("answer"))
print("\nPrompt preview:")
print(rag_result.get("prompt", "")[:800])
print("\nRetrieved docs:", len(rag_result.get("results", [])))


Status: 200
Answer: I do not know.

Prompt preview:
You are a helpful assistant. Answer the user question using only the provided context. If the context does not contain enough information, say you do not know.

Context:
[1] Title: Untitled
The economy is largely based in agriculture (making up 10% of the GDP and the most substantial export) and the state-sector, and relies heavily on world trade. Consequently, it is badly affected by any downturn in global prices. However, the economy is on the whole more stable than surrounding states, and it maintains a solid reputation with investors.

[2] Title: Untitled
Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its me

Retrieved docs: 3


## Evaluate Against QA Dataset

In [12]:
eval_payload = {
    "source": "hf://datasets/rag-datasets/rag-mini-wikipedia/data/test.parquet/part.0.parquet",
    "limit": 5,
    "top_k": 3,
}
status_code, eval_result = post_json("/rag/evaluate", eval_payload, timeout=600)
print("Status:", status_code)
print("Summary:", json.dumps(eval_result.get("summary", {}), indent=2))
print("First item:", json.dumps(eval_result.get("items", [])[0], indent=2) if eval_result.get("items") else "No items")


Status: 200
Summary: {
  "total": 5,
  "exact_match_rate": 0.0,
  "contains_expected_rate": 0.2
}
First item: {
  "question": "Was Abraham Lincoln the sixteenth President of the United States?",
  "expected_answer": "yes",
  "generated_answer": "I do not know.",
  "exact_match": false,
  "contains_expected": false
}


## Verify Relevant Endpoints

In [13]:
checks = [
    ("GET", "/health"),
    ("GET", "/vectordb/status"),
    ("POST", "/vectordb/query"),
    ("POST", "/rag/query"),
    ("POST", "/rag/evaluate"),
]

for method, endpoint in checks:
    print(f"{method} {endpoint}")


GET /health
GET /vectordb/status
POST /vectordb/query
POST /rag/query
POST /rag/evaluate


## Cleanup

In [14]:
if "server_proc" in globals() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except Exception:
        server_proc.kill()
print("Server stopped.")


Server stopped.


## Summary

Sprint 4 observability complete:
- Augmented prompt is generated from retrieved context
- GPT answer is returned by `/rag/query`
- Dataset quality metrics are produced by `/rag/evaluate`

## Validation

This section validates that:
- retrieved matches are used to construct an augmented prompt
- the augmented prompt is sent to the LLM and returns a synthesized answer
- quality and performance are measured using the HF test QA dataset

In [15]:
import statistics
import time
import urllib.request
from urllib.parse import urlencode

import pandas as pd


def ensure_server_running() -> None:
    global server_proc
    try:
        with urllib.request.urlopen(f"{BASE_URL}/health", timeout=2) as resp:
            if resp.status == 200:
                return
    except Exception:
        pass

    cmd_local = [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--host",
        "127.0.0.1",
        "--port",
        str(PORT),
    ]
    server_env_local = dict(os.environ)
    server_env_local["PYTHONPATH"] = str(SRC_DIR)
    server_proc = subprocess.Popen(cmd_local, cwd=str(ROOT), env=server_env_local)

    for _ in range(40):
        try:
            with urllib.request.urlopen(f"{BASE_URL}/health", timeout=2) as resp:
                if resp.status == 200:
                    return
        except Exception:
            time.sleep(0.5)

    raise RuntimeError("Server did not start for Validation section")


ensure_server_running()

# Ensure the vector DB has content for retrieval and prompt augmentation checks.
status_code, db_status = get_json("/vectordb/status")
if not db_status.get("initialized") or db_status.get("document_count", 0) == 0:
    status_code, _ = post_json(
        "/vectordb/ingest",
        {
            "source": "hf://datasets/rag-datasets/rag-mini-wikipedia/data/passages.parquet/part.0.parquet",
            "limit": 20,
        },
        timeout=300,
    )

validation_query = "What is artificial intelligence?"

# Step 1: Retrieve matches first.
status_code, retrieval_payload = post_json(
    "/vectordb/query",
    {"query": validation_query, "top_k": 3},
    timeout=180,
)
retrieved_docs = retrieval_payload.get("results", [])

context_snippets = []
for idx, doc in enumerate(retrieved_docs, start=1):
    snippet = (doc.get("text", "") or "")[:220].replace("\n", " ")
    context_snippets.append(f"[{idx}] {snippet}")

manual_augmented_prompt = (
    "You are a helpful assistant. Use only the provided context to answer.\n\n"
    f"Question: {validation_query}\n\n"
    "Context:\n"
    + "\n".join(context_snippets)
    + "\n\nAnswer:"
)

print("Retrieved matches:", len(retrieved_docs))
print("Manual augmented prompt preview:\n")
print(manual_augmented_prompt[:1000])

# Step 2: Send augmented prompt flow to LLM via /rag/query.
start = time.perf_counter()
status_code, rag_validation = post_json(
    "/rag/query",
    {"query": validation_query, "top_k": 3},
    timeout=240,
)
rag_latency_sec = time.perf_counter() - start

print("\nRAG endpoint status:", status_code)
print("RAG prompt preview:\n")
print((rag_validation.get("prompt") or "")[:1000])
print("\nSynthesized answer:\n")
print(rag_validation.get("answer", ""))
print(f"\nSingle-query latency: {rag_latency_sec:.2f}s")

# Step 3: Evaluate quality and performance with QA test set.
eval_limit = 25
start = time.perf_counter()
status_code, eval_validation = post_json(
    "/rag/evaluate",
    {
        "source": "hf://datasets/rag-datasets/rag-mini-wikipedia/data/test.parquet/part.0.parquet",
        "limit": eval_limit,
        "top_k": 3,
    },
    timeout=900,
)
eval_duration_sec = time.perf_counter() - start

summary = eval_validation.get("summary", {})
print("\nEvaluation status:", status_code)
print("Evaluation summary:", summary)
print(f"Evaluation runtime for {eval_limit} items: {eval_duration_sec:.2f}s")

# Additional latency sample using first questions from the same dataset.
qa_df = pd.read_parquet(
    "hf://datasets/rag-datasets/rag-mini-wikipedia/data/test.parquet/part.0.parquet"
).head(10)
question_col = "question" if "question" in qa_df.columns else qa_df.columns[0]
latencies = []

for question in qa_df[question_col].astype(str).tolist():
    t0 = time.perf_counter()
    _status, _payload = post_json("/rag/query", {"query": question, "top_k": 3}, timeout=240)
    latencies.append(time.perf_counter() - t0)

latencies_sorted = sorted(latencies)
p95_idx = max(int(len(latencies_sorted) * 0.95) - 1, 0)
print("\nLatency sample (10 QA questions):")
print(f"avg={statistics.mean(latencies):.2f}s min={min(latencies):.2f}s max={max(latencies):.2f}s p95={latencies_sorted[p95_idx]:.2f}s")

Retrieved matches: 3
Manual augmented prompt preview:

You are a helpful assistant. Use only the provided context to answer.

Question: What is artificial intelligence?

Context:
[1] The Plaza Independencia ("Independence Square"), in Montevideo, hosts the tomb of JosÃ© Artigas, late leader of the Provincia Oriental and the Liga Federal. In front of the square, the Palacio Salvo can be seen.
[2] * "River of colorful or 'painted' chinchillas (birds)": poetic interpretation attributed to Juan Zorrilla de San MartÃ­n. 
[3] The Uruguayans' road to independence was much longer than those of other countries in the Americas. Early efforts at attaining independence focused on overthrow of Spanish rule, a process begun by Jose Gervasio Artigas i

Answer:

RAG endpoint status: 200
RAG prompt preview:

You are a helpful assistant. Answer the user question using only the provided context. If the context does not contain enough information, say you do not know.

Context:
[1] Title: Untitled
The Pla

c:\Users\mlamar\Working Directory(Local)\Forge\m2\da-rag-project-2026_mlamar2019_2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Latency sample (10 QA questions):
avg=5.04s min=0.69s max=21.85s p95=17.92s


## Wrap Up

Validation demonstrates Sprint 4 success criteria:
- retrieved documents are surfaced and used for prompt augmentation
- `/rag/query` returns both augmented prompt and synthesized LLM answer
- `/rag/evaluate` reports quality metrics using the HF test QA dataset
- latency metrics provide a basic performance profile for end-to-end RAG

In [16]:
quality = eval_validation.get("summary", {}) if "eval_validation" in globals() else {}
exact_rate = quality.get("exact_match_rate", 0.0)
contains_rate = quality.get("contains_expected_rate", 0.0)

print("Validation Wrap Up")
print("------------------")
print(f"single_query_latency_sec: {rag_latency_sec:.2f}" if "rag_latency_sec" in globals() else "single_query_latency_sec: n/a")
print(f"qa_eval_exact_match_rate: {exact_rate:.3f}")
print(f"qa_eval_contains_expected_rate: {contains_rate:.3f}")

quality_ok = contains_rate >= 0.3
latency_ok = ("latencies" in globals() and statistics.mean(latencies) <= 3.0)

print("\nQuality check:", "PASS" if quality_ok else "NEEDS IMPROVEMENT")
print("Performance check:", "PASS" if latency_ok else "NEEDS IMPROVEMENT")

if "server_proc" in globals() and server_proc is not None and server_proc.poll() is None:
    server_proc.terminate()
    print("\nServer cleanup: terminated")
else:
    print("\nServer cleanup: not running")

Validation Wrap Up
------------------
single_query_latency_sec: 0.92
qa_eval_exact_match_rate: 0.000
qa_eval_contains_expected_rate: 0.080

Quality check: NEEDS IMPROVEMENT
Performance check: NEEDS IMPROVEMENT

Server cleanup: terminated
